# 02 - Label Standardization and Business Mapping

This notebook standardizes labels and maps Banking77 intents into a business-friendly taxonomy.
It also prepares sentiment labels for consistent downstream training.


## Setup — run this once before anything else


**Create your key file.** In the same folder as this notebook, copy api_keys.json.example to a new file named api_keys.json and paste your real Groq / Gemini keys into it. api_keys.json is never printed or embedded in this notebook — keep it out of git (add it to .gitignore).

**used API keys** = gemini-2.5-flash and openai/gpt-oss-120b

In [1]:
import os, re, json, time, random
from pathlib import Path
from typing import Any, Dict, List, Optional
import pandas as pd
from groq import Groq
from google import genai
from google.genai import types as genai_types
from tqdm.notebook import tqdm

In [ ]:
API_KEYS_PATH = Path("api_keys.json")


def usable_keys(values):
    return [
        str(value).strip()
        for value in values
        if str(value).strip() and "PASTE_" not in str(value)
    ]


def load_api_keys(path: Path = API_KEYS_PATH) -> Dict[str, List[str]]:
    if not path.exists():
        raise FileNotFoundError(
            f"'{path}' not found next to this notebook.\n"
            f"Copy 'api_keys.json.example' to 'api_keys.json' and fill in your real keys."
        )
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return {
        "groq":   usable_keys(data.get("groq_keys", [])),
        "gemini": usable_keys(data.get("gemini_keys", [])),
    }


_keys       = load_api_keys()
groq_keys   = _keys["groq"]
gemini_keys = _keys["gemini"]

INPUT_CSV      = "data/processed/scraped_reviews_clean_combined.csv"
OUTPUT_CSV     = "data/labeled/labeled_reviews.csv"
CHECKPOINT_CSV = "data/labeled/labeled_reviews_checkpoint.csv"
ERROR_LOG_CSV  = "data/labeled/labeling_errors.csv"

MODEL_NAME_GROQ   = "openai/gpt-oss-120b"

MODEL_NAME_GEMINI = "gemini-2.5-flash"

BATCH_SIZE        = 8
MAX_RETRIES       = 5
BASE_SLEEP        = 2
MAX_SLEEP         = 30

TEXT_COL_CANDIDATES = ["review_text", "comment", "text", "review"]

_providers = [
    *[
        {"name": f"groq_{index}", "key": key, "cooldown_until": 0.0}
        for index, key in enumerate(groq_keys, start=1)
    ],
    *[
        {"name": f"gemini_{index}", "key": key, "cooldown_until": 0.0}
        for index, key in enumerate(gemini_keys, start=1)
    ],
]
_current_provider_index = 0

if not _providers:
    raise RuntimeError(
        "No usable API keys found in api_keys.json. "
        "Add at least one groq_keys or gemini_keys entry."
    )
print("Configured providers:", ", ".join(p["name"] for p in _providers))

Configured providers: groq_1, groq_2, groq_3, groq_4, gemini_1, gemini_2


In [3]:
def get_next_provider() -> Optional[Dict]:
    """Return the next available provider (not in cooldown). Returns None if all are cooling down."""
    global _current_provider_index
    now = time.time()
    for _ in range(len(_providers)):
        p = _providers[_current_provider_index % len(_providers)]
        _current_provider_index += 1
        if now >= p["cooldown_until"]:
            return p
    return None   # all providers in cooldown


def set_cooldown(provider: Dict, seconds: float) -> None:
    provider["cooldown_until"] = time.time() + seconds
    print(f"  ⏳ [{provider['name']}] Cooling down for {seconds:.0f}s "
          f"(until {time.strftime('%H:%M:%S', time.localtime(provider['cooldown_until']))})")



In [ ]:
import json as _json

# SENTIMENT

SENTIMENT_LABELS = {
    "positive": {
        "code": "POS",
        "description": "Customer expresses satisfaction, appreciation, or delight.",
        "churn_risk":      "LOW",
        "review_signal":   "Retention / NPS booster",
        "default_action":  "Archive for marketing reuse",
    },
    "neutral": {
        "code": "NEU",
        "description": "Informational, ambiguous, or no strong emotional valence.",
        "churn_risk":      "MEDIUM",
        "review_signal":   "Feature request or passive feedback",
        "default_action":  "Route to product backlog",
    },
    "negative": {
        "code": "NEG",
        "description": "Customer expresses dissatisfaction, frustration, or complaint.",
        "churn_risk":      "HIGH",
        "review_signal":   "Churn trigger / regulatory exposure",
        "default_action":  "Escalate within SLA",
    },
    "mixed": {
        "code": "MIX",
        "description": "Review contains both positive and negative signals.",
        "churn_risk":      "MEDIUM",
        "review_signal":   "Partial satisfaction — watch for escalation",
        "default_action":  "Flag for product and CX review",
    },
}

# INTENT  

INTENT_LABELS = {

    "technical_failure": {
        "code": "TECH", "priority": "P1", "sla_hours": 24,
        "description": (
            "App or backend technically broken: crashes, force-close, "
            "OTP not delivered, API/server error, notification failure, "
            "UI regression after update. Core functionality does not work."
        ),
        "action_owner": "Engineering + Customer Support",
        "escalation_note": "Assign to on-call engineer. Check Sentry / crash dashboard.",
        "v1_equivalent": "complaint (app/backend/OTP broken)",
    },
    "access_blocked": {
        "code": "ACBL", "priority": "P1", "sla_hours": 24,
        "description": (
            "Customer cannot get into the app: login failure, biometric failure, "
            "password reset broken, device limit exceeded, KYC block, session "
            "expiry loop, developer-options restriction."
        ),
        "action_owner": "Engineering + Identity/Auth Team",
        "escalation_note": "If KYC-related escalate to Compliance. Log device fingerprint.",
        "v1_equivalent": "complaint (login/auth/device blocked)",
    },
    "payment_failure": {
        "code": "PAYF", "priority": "P1", "sla_hours": 12,
        "description": (
            "A payment, transfer, top-up, QR, or bill payment initiated by the "
            "customer did not complete or shows in an error/stuck state. "
            "Includes LankaPay / CEFT / SLIPS rail failures and QR failures."
        ),
        "action_owner": "Engineering + Finance + Payments Team",
        "escalation_note": "Cross-check with CBS/core banking logs. CBSL reportable if stuck > 24h.",
        "v1_equivalent": "complaint (payment/api_failure)",
    },
    "data_integrity_issue": {
        "code": "DATA", "priority": "P1", "sla_hours": 24,
        "description": (
            "Customer sees wrong balance, missing or duplicate transaction, "
            "incorrect statement, or data that does not match what the bank holds."
        ),
        "action_owner": "Engineering + Core Banking + Finance",
        "escalation_note": "Potential CBSL reporting obligation. Escalate to CRO if balance discrepancy > LKR 10,000.",
        "v1_equivalent": "complaint (data_sync_issue)",
    },
    "service_quality": {
        "code": "SVCQ", "priority": "P2", "sla_hours": 48,
        "description": (
            "App technically works but is slow, laggy, unresponsive, times out, "
            "or is painful to use due to performance degradation."
        ),
        "action_owner": "Engineering / DevOps",
        "escalation_note": "Check infra metrics for same time window. Correlate with deploy events.",
        "v1_equivalent": "complaint (slow) + performance_issue",
    },
    "support_failure": {
        "code": "SUPF", "priority": "P2", "sla_hours": 48,
        "description": (
            "Customer explicitly says support ignored them, gave no reply, "
            "was unhelpful after multiple contacts, or the complaint channel itself "
            "is broken."
        ),
        "action_owner": "Head of Customer Experience + CRO",
        "escalation_note": "Pattern of support failures is a regulatory risk. Flag to CRO if 3+ in same week.",
        "v1_equivalent": "complaint (unresponsive_support)",
    },

    "interbank_transfer_failure": {
        "code": "IBTF", "priority": "P1", "sla_hours": 12,
        "description": (
            "Transfer to another Sri Lankan bank failed or is stuck — specifically "
            "LankaPay, CEFT, SLIPS, or RTGS rail. Distinct from internal transfers "
            "within the same bank."
        ),
        "action_owner": "Payments Team + LankaPay Liaison + Engineering",
        "escalation_note": "CBSL has mandatory reporting for interbank settlement failures. Log immediately.",
        "v1_equivalent": "NEW — was previously lost inside complaint/api_failure",
    },
    "agent_banking_issue": {
        "code": "AGNT", "priority": "P2", "sla_hours": 48,
        "description": (
            "Problem related to agent banking: field agent not available, agent "
            "gave wrong information, agent machine down, Sampath Vishwa / BOC agent "
            "network failure, or mismatch between agent record and app record."
        ),
        "action_owner": "Agent Network Operations + Regional Manager",
        "escalation_note": "Rural customers disproportionately affected. Escalate if cluster of reports from same region.",
        "v1_equivalent": "NEW — previously undetected, fell into noise or general complaint",
    },
    "fx_conversion_complaint": {
        "code": "FXCC", "priority": "P2", "sla_hours": 48,
        "description": (
            "Complaint about forex rate, remittance, foreign currency conversion, "
            "or multi-currency wallet. Includes diaspora remittance to Sri Lanka "
            "and disagreement on exchange rate applied."
        ),
        "action_owner": "Treasury + Finance + Compliance",
        "escalation_note": "Any FX dispute > USD 500 equivalent requires Compliance review under CBSL FX rules.",
        "v1_equivalent": "NEW — previously fell into refund_request or noise",
    },


    "ux_friction": {
        "code": "UXFR", "priority": "P2", "sla_hours": 72,
        "description": "Usability or flow friction — app works but is hard to use, confusing, or poorly designed.",
        "action_owner": "UX + Product",
        "escalation_note": "Group by screen/flow for sprint planning.",
    },


    "refund_request": {
        "code": "RFND", "priority": "P1", "sla_hours": 24,
        "description": "Customer asks for refund, reversal, or money back.",
        "action_owner": "Finance + Customer Support",
        "escalation_note": "Log for CBSL dispute reporting if unresolved.",
    },
    "security_concern": {
        "code": "SECU", "priority": "P0", "sla_hours": 4,
        "description": "Possible security, fraud, or privacy problem.",
        "action_owner": "CISO + CRO",
        "escalation_note": "Immediate CRO notification required.",
    },
    "regulatory_risk": {
        "code": "REGU", "priority": "P0", "sla_hours": 4,
        "description": "Compliance, legal, or regulatory concern.",
        "action_owner": "Legal + CRO",
        "escalation_note": "Must be logged before end of business day.",
    },
    "complaint": {
        "code": "CMPL", "priority": "P1", "sla_hours": 24,
        "description": "Broken feature, failed transaction, or blocked access.",
        "action_owner": "Engineering + Customer Support",
        "escalation_note": "Assign to on-call engineer if P0 root cause.",
    },


    "churn_signal": {
        "code": "CHRN", "priority": "P1", "sla_hours": 12,
        "description": "Customer threatens to leave or switch provider.",
        "action_owner": "Retention Team + Head of Product",
        "escalation_note": "Trigger win-back flow within SLA.",
    },
    "competitor_switch": {
        "code": "CPSW", "priority": "P1", "sla_hours": 12,
        "description": "Customer explicitly states they are moving to a named competitor.",
        "action_owner": "Retention Team + Marketing",
        "escalation_note": "Log competitor name in competitive intelligence tracker.",
    },
    "account_closure": {
        "code": "ACLO", "priority": "P1", "sla_hours": 12,
        "description": "Customer wants to close or deactivate account.",
        "action_owner": "Retention Team + Compliance",
        "escalation_note": "Regulatory obligation to log closure intent.",
    },

    "feature_request": {
        "code": "FREQ", "priority": "P3", "sla_hours": 168,
        "description": "Customer requests a new capability or improvement.",
        "action_owner": "Product Management",
        "escalation_note": "Tag with feature area for roadmap clustering.",
    },

    "praise": {
        "code": "PRAI", "priority": "P4", "sla_hours": None,
        "description": "Positive experience or endorsement — no action needed.",
        "action_owner": "Marketing / CX",
        "escalation_note": "Extract for NPS and testimonials.",
    },

    "noise": {
        "code": "NOIS", "priority": "P5", "sla_hours": None,
        "description": "Gibberish, spam, off-topic, or unreadable text.",
        "action_owner": "Data Engineering",
        "escalation_note": "Exclude from ML training set.",
    },
}


# ROOT CAUSE  

ROOT_CAUSE_LABELS = {

    # Access & Authentication 

    "authentication_failure": {
        "domain": "Access Control",
        "cro_flag": True,
        "description": (
            "Login failure, password reset failure, or session expires repeatedly. "
            "Does NOT cover biometric (see biometric_failure) or "
            "OTP issues (see notification_delivery_failure / otp_loop_failure)."
        ),
        "review_example": "Cannot log in even with correct password",
    },
    "biometric_failure": {
        "domain": "Access Control",
        "cro_flag": True,
        "description": (
            "Fingerprint or face ID does not work in the app — either refuses "
            "to scan, loops, or reverts to PIN without explanation. "
            "Split from authentication_failure for precise routing to mobile security team."
        ),
        "review_example": "Fingerprint won't work, keeps asking for PIN",
        "v1_equivalent": "NEW — previously lumped into authentication_failure",
    },
    "device_limit_exceeded": {
        "domain": "Access Control",
        "cro_flag": True,
        "description": "Blocked by registered-device limit or device policy.",
        "review_example": "Exceeded maximum device registrations",
    },

    # OTP / Notification
    "notification_delivery_failure": {
        "domain": "Platform Reliability",
        "cro_flag": True,
        "description": (
            "OTP or verification SMS is never received at all — "
            "does not arrive even after retries. "
            "Distinct from otp_loop_failure where OTP arrives but is rejected."
        ),
        "review_example": "I never receive the OTP SMS",
    },
    "otp_loop_failure": {
        "domain": "Platform Reliability",
        "cro_flag": True,
        "description": (
            "OTP arrives correctly but the app rejects it, calls it expired "
            "immediately, or loops the OTP screen endlessly. "
            "Different from notification_delivery_failure (OTP not received)."
        ),
        "review_example": "OTP comes but app says invalid / expired immediately",
        "v1_equivalent": "NEW — previously lumped into notification_delivery_failure",
    },

    # Platform Reliability 

    "slow_performance": {
        "domain": "Platform Reliability",
        "cro_flag": False,
        "description": "App speed, latency, timeout, or lag problems — app still works.",
        "review_example": "App takes 30 seconds to load",
    },
    "app_crash_or_bug": {
        "domain": "Platform Reliability",
        "cro_flag": False,
        "description": (
            "App crashes, freezes, or has a persistent UI bug — NOT caused by "
            "a recent update. If the crash started after an update use app_update_regression."
        ),
        "review_example": "App crashes every time I open it",
    },
    "app_update_regression": {
        "domain": "Platform Reliability",
        "cro_flag": True,
        "description": (
            "App worked before but broke after a specific version update. "
            "Signals a regression introduced by the release team. "
            "CRO flag true because mass regressions generate surge in negative reviews "
            "and may affect CBS integration."
        ),
        "review_example": "After the latest update the app won't open at all",
        "v1_equivalent": "NEW — split from app_crash_or_bug for release management routing",
    },

    # Payments & Transfers

    "api_failure": {
        "domain": "Backend / Integration",
        "cro_flag": True,
        "description": (
            "Internal API call fails, backend endpoint errors, or service "
            "unavailable — for transfers or payments within the same bank's system."
        ),
        "review_example": "Payment fails with 'server error' on internal transfer",
    },
    "lankapay_ceft_failure": {
        "domain": "Interbank Settlement",
        "cro_flag": True,
        "description": (
            "Transfer via LankaPay, CEFT, or SLIPS rail fails or gets stuck. "
            "Distinct from api_failure (internal) — this involves the national "
            "payment infrastructure. CBSL-reportable."
        ),
        "review_example": "Transfer to Sampath Bank never arrived — deducted from my account",
        "v1_equivalent": "NEW — previously classified as api_failure, losing CBSL escalation signal",
    },
    "qr_payment_failure": {
        "domain": "Payments",
        "cro_flag": True,
        "description": (
            "Lanka QR or in-app QR code scan payment fails — QR does not scan, "
            "payment does not process after scan, or merchant does not receive funds."
        ),
        "review_example": "QR code payment failed at the supermarket — was embarrassing",
        "v1_equivalent": "NEW — previously lost in api_failure or app_crash_or_bug",
    },

    # Backend / Data 

    "data_sync_issue": {
        "domain": "Backend / Data",
        "cro_flag": True,
        "description": "Balances, transactions, or records do not sync or display correctly.",
        "review_example": "My balance shows wrong amount after transfer",
    },

    # Security & Compliance

    "security_restriction_ux": {
        "domain": "Security Policy",
        "cro_flag": True,
        "description": "Legitimate security control causes access friction or user block.",
        "review_example": "Have to turn off developer options just to open the app",
    },
    "poor_security_implementation": {
        "domain": "Security Engineering",
        "cro_flag": True,
        "description": "Weak, broken, or band-aid security identified by customer.",
        "review_example": "Their fix just breaks USB debugging — not real security",
    },
    "regulatory_block": {
        "domain": "Compliance / Regulation",
        "cro_flag": True,
        "description": (
            "Issue caused by compliance rules, KYC, or regulatory checks — "
            "and the customer KNOWS the reason (e.g. 'my KYC failed'). "
            "If customer says account frozen with NO reason given, "
            "use account_freeze_no_reason instead."
        ),
        "review_example": "Account frozen — must be KYC issue",
    },
    "account_freeze_no_reason": {
        "domain": "Compliance / Trust",
        "cro_flag": True,
        "description": (
            "Account frozen, blocked, or restricted and the customer was given "
            "NO explanation. Creates regulatory and reputational risk because "
            "CBSL requires banks to communicate account action reasons to customers."
        ),
        "review_example": "Account blocked without any reason or notice — can't access my money",
        "v1_equivalent": "NEW — previously misclassified as authentication_failure or regulatory_block",
    },

    # Billing & Revenue

    "unjustified_billing": {
        "domain": "Revenue / Trust",
        "cro_flag": True,
        "description": "Unexpected charge, fee dispute, or billing complaint (general).",
        "review_example": "Charged annual fee for an app that doesn't work",
    },
    "fee_without_service": {
        "domain": "Revenue / Trust",
        "cro_flag": True,
        "description": (
            "Customer was charged a fee but the service they paid for was "
            "never delivered or was broken. More specific than unjustified_billing — "
            "the fee is for a transaction that failed. CBSL consumer protection applicable."
        ),
        "review_example": "Charged Rs.50 transaction fee but the transfer failed",
        "v1_equivalent": "NEW — previously lost inside unjustified_billing",
    },
    "misleading_promotion": {
        "domain": "Marketing / Trust",
        "cro_flag": True,
        "description": (
            "Advertised offer, cashback, or promotion was not honoured as described. "
            "Includes bait-and-switch offers and fine-print exclusions not made clear. "
            "CRO flag true: CBSL monitors misleading financial advertising."
        ),
        "review_example": "Promised 10% cashback but only got 2% — nowhere does it say that limit",
        "v1_equivalent": "NEW — previously fell into feature_request or general_satisfaction",
    },

    # Agent Banking 

    "agent_network_failure": {
        "domain": "Agent Banking",
        "cro_flag": False,
        "description": (
            "Field agent unavailable, agent machine/terminal down, agent gave "
            "incorrect information, or mismatch between agent record and app record. "
            "Relevant to Sampath Vishwa, BOC Mobile Banking, and other agent networks."
        ),
        "review_example": "Agent in my area is never available — nearest one is 10km away",
        "v1_equivalent": "NEW — previously fell into noise or general complaint",
    },

    # FX / Remittance 

    "fx_rate_dispute": {
        "domain": "Treasury / FX",
        "cro_flag": True,
        "description": (
            "Customer disputes the forex rate applied, remittance conversion, "
            "or multi-currency wallet rate. Relevant for diaspora inward remittances "
            "and foreign card transactions processed in Sri Lanka."
        ),
        "review_example": "Applied a terrible exchange rate on my remittance — lost Rs.5000",
        "v1_equivalent": "NEW — previously fell into refund_request or noise",
    },

    # Customer Service

    "unresponsive_support": {
        "domain": "Customer Experience",
        "cro_flag": False,
        "description": "Support is slow, ignored, unhelpful, or unresponsive.",
        "review_example": "Contacted support 5 times — no response",
    },

    # Competitive

    "competitor_reference": {
        "domain": "Competitive Intelligence",
        "cro_flag": True,
        "description": "Customer explicitly names or recommends a competing app or bank.",
        "review_example": "HNB app is much better — switching to them",
    },

    # Positive Drivers 

    "offer_discoverability": {
        "domain": "Product Value",
        "cro_flag": False,
        "description": "Customer appreciates easy access to offers or useful features.",
        "review_example": "I can now find offers easily",
    },
    "financial_safety_confidence": {
        "domain": "Trust & Safety",
        "cro_flag": False,
        "description": "Customer explicitly trusts the app for safe financial transactions.",
        "review_example": "Always keeps checking — feels very safe",
    },
    "general_satisfaction": {
        "domain": "Brand Perception",
        "cro_flag": False,
        "description": "Overall positive experience without a specific feature driver.",
        "review_example": "Best banking app in Sri Lanka",
    },

    #  Noise

    "unreadable_text": {
        "domain": "Data Quality",
        "cro_flag": False,
        "description": "Review is gibberish, emoji-only, random characters, or spam.",
        "review_example": "aasdfghjkl 😂😂😂",
    },

    # Fallback 

    "other": {
        "domain": "Unclassified",
        "cro_flag": False,
        "description": "Does not fit any defined root cause — use sparingly.",
        "review_example": "Edge case not covered by taxonomy",
    },
}

# RISK MATRIX  


CRO_RISK_MATRIX = {
    # Critical (10) — regulatory / security 
    ("security_concern",   "poor_security_implementation"):      10,
    ("security_concern",   "regulatory_block"):                  10,
    ("regulatory_risk",    "regulatory_block"):                  10,
    ("regulatory_risk",    "account_freeze_no_reason"):          10,  
    ("data_integrity_issue", "data_sync_issue"):                  9,  
    ("payment_failure",    "lankapay_ceft_failure"):              9,  
    ("interbank_transfer_failure", "lankapay_ceft_failure"):      9,  
    
    ("security_concern",   "security_restriction_ux"):            9,
    ("regulatory_risk",    "unjustified_billing"):                9,
    ("regulatory_risk",    "fee_without_service"):                9,  
    ("regulatory_risk",    "misleading_promotion"):               8,  

    ("churn_signal",       "competitor_reference"):               8,
    ("competitor_switch",  "competitor_reference"):               8,
    ("account_closure",    "competitor_reference"):               8,
    ("refund_request",     "unjustified_billing"):                8,
    ("refund_request",     "fee_without_service"):                8,  
    ("access_blocked",     "authentication_failure"):             8,  
    ("access_blocked",     "biometric_failure"):                  8,  
    ("access_blocked",     "device_limit_exceeded"):              8,  
    ("access_blocked",     "regulatory_block"):                   8,  
    ("access_blocked",     "account_freeze_no_reason"):           9,  
    ("payment_failure",    "api_failure"):                        8,  
    ("payment_failure",    "qr_payment_failure"):                 8,  
    ("payment_failure",    "fee_without_service"):                8,  
    ("data_integrity_issue", "lankapay_ceft_failure"):            8,  
    ("technical_failure",  "notification_delivery_failure"):      8,  
    ("technical_failure",  "otp_loop_failure"):                   8,  
    ("technical_failure",  "api_failure"):                        8,  
    ("fx_conversion_complaint", "fx_rate_dispute"):               7,  
   
    ("technical_failure",  "app_update_regression"):              7,  
    ("access_blocked",     "otp_loop_failure"):                   7,  
    ("agent_banking_issue","agent_network_failure"):              6,  
    
    ("support_failure",    "unresponsive_support"):               6,  
    ("technical_failure",  "app_crash_or_bug"):                   6,  
    ("ux_friction",        "security_restriction_ux"):            6,
    ("ux_friction",        "app_update_regression"):              6,  
    
    ("service_quality",    "slow_performance"):                   5,  
    ("technical_failure",  "app_crash_or_bug"):                   5,
    
    ("ux_friction",        "slow_performance"):                   4,
    ("feature_request",    "app_update_regression"):              4,  
  
    ("feature_request",    "offer_discoverability"):              3,
    ("feature_request",    "slow_performance"):                   3,
    ("ux_friction",        "offer_discoverability"):              3,
   
    ("praise",             "general_satisfaction"):               1,
    ("praise",             "financial_safety_confidence"):        1,
    ("praise",             "offer_discoverability"):              1,
   
    ("noise",              "unreadable_text"):                    0,
}

DEFAULT_RISK_SCORE = 3

VALID_SENTIMENTS  = list(SENTIMENT_LABELS.keys())
VALID_INTENTS     = list(INTENT_LABELS.keys())
VALID_ROOT_CAUSES = list(ROOT_CAUSE_LABELS.keys())

RISK_BANDS = {
    (0, 2):  {"band": "NEGLIGIBLE", "action": "Archive / marketing use"},
    (3, 4):  {"band": "LOW",        "action": "Backlog — next sprint"},
    (5, 6):  {"band": "MODERATE",   "action": "Assign to Product team this week"},
    (7, 8):  {"band": "HIGH",       "action": "Escalate to Head of Product within 24h"},
    (9, 10): {"band": "CRITICAL",   "action": "Immediate CRO + Legal escalation"},
}


In [5]:
def get_risk_band(score):
    for (lo, hi), v in RISK_BANDS.items():
        if lo <= score <= hi:
            return {"score": score, **v}
    return {"score": score, "band": "UNKNOWN", "action": "Manual review required"}


def validate_label(field, value):
    field = str(field).lower().strip()
    value = str(value).lower().strip()
    return value in {"sentiment": VALID_SENTIMENTS, "intent": VALID_INTENTS,
                     "root_cause": VALID_ROOT_CAUSES}.get(field, [])


def score_risk(intent, root_cause, sentiment=None):
    base = CRO_RISK_MATRIX.get((str(intent).lower().strip(),
                                 str(root_cause).lower().strip()), DEFAULT_RISK_SCORE)
    if str(sentiment).lower().strip() == "negative":
        base += 1
    return max(0, min(10, int(base)))


def normalize_labels(sentiment, intent, root_cause, risk_score=None):
    s = str(sentiment).lower().strip()
    i = str(intent).lower().strip()
    r = str(root_cause).lower().strip()
    if s not in VALID_SENTIMENTS:  s = "neutral"
    if i not in VALID_INTENTS:     i = "noise"
    if r not in VALID_ROOT_CAUSES: r = "other"
    if risk_score is None:
        rs = score_risk(i, r, s)
    else:
        try:    rs = int(risk_score)
        except: rs = score_risk(i, r, s)
    rs = max(0, min(10, rs))
    band = get_risk_band(rs)
    return {"sentiment": s, "intent": i, "root_cause": r, "risk_score": rs,
            "risk_band": band["band"], "recommended_action": band["action"]}


In [1]:
# SYSTEM PROMPT

CRO_SYSTEM_PROMPT = f"""You are the Chief Risk Officer (CRO) assistant for a Sri Lankan financial services company.

Your job: analyze customer app reviews and produce CRO-grade labels for the company's
fintech apps (ComBank Digital, FriMi, Peoples Pay, Keells Pay, FlexPay, etc.).

IMPORTANT CONTEXT — SRI LANKA BANKING ENVIRONMENT:
  • LankaPay / CEFT / SLIPS: national interbank payment rails. Failures here are CBSL-reportable.
  • Lanka QR: national QR payment standard. Failures embarrass customers in public.
  • CBSL (Central Bank of Sri Lanka): regulator. Any compliance, KYC, or settlement issue must be flagged.
  • Agent banking: Sampath Vishwa, BOC agents — serve rural customers who cannot visit branches.
  • Common review languages: English, Sinhala (transliterated), Tamil (transliterated) — read intent not grammar.

CHAIN-OF-THOUGHT — apply in this order for EVERY review:

  Step 1 → Emotional state of the customer → sentiment
  Step 2 → What SPECIFIC action does the customer want or what SPECIFIC problem occurred? → intent
  Step 3 → What is the UNDERLYING technical or operational failure? → root_cause
  Step 4 → Look up (intent, root_cause) in the risk matrix → base risk_score
           Add +1 if sentiment = negative AND risk_score < 10.

MULTI-ISSUE RULE: If a review contains multiple problems, pick the HIGHEST-RISK
root_cause. NEVER assign root_cause = other if ANY specific cause matches below.


"complaint" is now split into six precise intents:
  technical_failure | access_blocked | payment_failure |
  data_integrity_issue | service_quality | support_failure
Choose whichever BEST describes what happened.

LABELING DECISION RULES (top-to-bottom; first match wins)

RULE 1 — NOISE / SPAM
  If review is gibberish, emoji-only, unreadable, or off-topic:
    intent = noise | root_cause = unreadable_text | risk_score = 0

RULE 2 — PURE PRAISE
  If review is entirely positive with NO complaint, NO request, NO suggestion:
    sentiment = positive | intent = praise
    root_cause = general_satisfaction OR financial_safety_confidence OR offer_discoverability
    risk_score = 0, 1, or 2

RULE 3 — REGULATORY / COMPLIANCE BLOCK  [CBSL-reportable]
  If customer mentions: account frozen, account suspended, blocked by bank,
  CBSL, Central Bank, compliance hold, KYC failed, identity verification blocked
  account, source of funds check, regulatory requirement blocked access:
    intent = regulatory_risk | root_cause = regulatory_block | risk_score = 10

RULE 4 — ACCOUNT FROZEN WITHOUT REASON  [distinct from Rule 3]
  If account is frozen/blocked/restricted AND no reason was given or communicated
  to the customer (they do NOT mention KYC, CBSL, compliance themselves):
    intent = regulatory_risk | root_cause = account_freeze_no_reason | risk_score = 10
  WHY: CBSL requires banks to communicate account action reasons. Silence is a violation.

RULE 5 — OTP/SMS NEVER ARRIVES
  If customer says OTP, SMS, verification code, or push alert was never received
  (did not arrive at all even after retrying):
    intent = technical_failure | root_cause = notification_delivery_failure | risk_score = 8

RULE 6 — OTP ARRIVES BUT APP REJECTS IT  [distinct from Rule 5]
  If customer says OTP arrives on their phone but the app says "invalid" or "expired"
  immediately, or the OTP screen loops endlessly:
    intent = technical_failure | root_cause = otp_loop_failure | risk_score = 8

RULE 7 — SECURITY / DEVELOPER OPTIONS
  If customer must disable developer options, USB debugging, or any OS-level
  security setting just to open or use the app:
    intent = security_concern | root_cause = security_restriction_ux | risk_score = 9

RULE 8 — EXPLICIT SECURITY FLAW
  If customer says security is broken, fake, a band-aid, inadequate, or identifies
  a specific vulnerability:
    intent = security_concern | root_cause = poor_security_implementation | risk_score = 10

RULE 9 — COMPETITOR / SWITCHING
  If customer names a competing bank or app favourably OR explicitly says switching/switched:
    intent = competitor_switch | root_cause = competitor_reference | risk_score = 8
  If customer expresses frustration and hints at leaving (no competitor named):
    intent = churn_signal | root_cause = competitor_reference | risk_score = 8

RULE 10 — ACCOUNT CLOSURE
  If customer explicitly asks to close, deactivate, or permanently delete account:
    intent = account_closure
    root_cause = pick PRIMARY reason: competitor_reference | authentication_failure |
                 unresponsive_support | unjustified_billing | fee_without_service
    risk_score = 8

RULE 11 — REFUND / BILLING DISPUTE
  If customer asks for refund, reversal, or chargeback:
    intent = refund_request | root_cause = unjustified_billing | risk_score = 8
  If customer was charged a fee but the transaction/service failed:
    intent = refund_request | root_cause = fee_without_service | risk_score = 8

RULE 12 — MISLEADING PROMOTION / OFFER NOT HONOURED  [SL-specific]
  If advertised cashback, reward, or offer was not applied as described:
    intent = regulatory_risk | root_cause = misleading_promotion | risk_score = 8
  WHY: CBSL monitors misleading financial advertising. This is not a UX issue.

RULE 13 — LANKAPAY / CEFT / SLIPS / INTERBANK TRANSFER FAILURE  [SL-specific]
  If transfer to ANOTHER bank (Sampath, HNB, BOC, Peoples, NTB, etc.) failed,
  was deducted but not received, or is stuck with no reversal:
    intent = interbank_transfer_failure | root_cause = lankapay_ceft_failure | risk_score = 9
  WHY: National payment rail failure. CBSL mandatory reporting. Distinct from
  internal API failure within the same bank.

RULE 14 — QR PAYMENT FAILURE  [SL-specific]
  If Lanka QR or in-app QR payment fails during scan, or merchant did not receive funds:
    intent = payment_failure | root_cause = qr_payment_failure | risk_score = 8

RULE 15 — INTERNAL PAYMENT / TRANSFER / TOP-UP FAILURE
  If payment, transfer, or top-up within the same bank fails with a server/gateway error:
    intent = payment_failure | root_cause = api_failure | risk_score = 8

RULE 16 — WRONG BALANCE / MISSING TRANSACTION / DATA ERROR
  If customer reports wrong balance, missing or duplicate transaction, or incorrect statement:
    intent = data_integrity_issue | root_cause = data_sync_issue | risk_score = 9

RULE 17 — AGENT BANKING ISSUE  [SL-specific]
  If customer mentions agent banking, field agent, Sampath Vishwa, agent terminal,
  or an agent gave wrong information / agent not available:
    intent = agent_banking_issue | root_cause = agent_network_failure | risk_score = 6

RULE 18 — FOREX / REMITTANCE / FX RATE COMPLAINT  [SL-specific]
  If customer disputes foreign exchange rate, remittance conversion, or multi-currency
  wallet rate applied by the app:
    intent = fx_conversion_complaint | root_cause = fx_rate_dispute | risk_score = 7

RULE 19 — APP BROKE AFTER UPDATE  [distinct from plain crash]
  If customer says app worked before but broke specifically after an update/upgrade:
    intent = technical_failure | root_cause = app_update_regression | risk_score = 7
  If financial feature (payments, transfers) is broken after update:
    intent = payment_failure | root_cause = app_update_regression | risk_score = 8

RULE 20 — APP CRASH / BUG (not update-related)
  If customer says app crashes, freezes, or force-closes with no mention of update:
    intent = technical_failure | root_cause = app_crash_or_bug | risk_score = 6

RULE 21 — SLOW / LAG / TIMEOUT (app works but is painful)
  If customer says app is slow, laggy, takes too long to load, times out, or hangs:
    intent = service_quality | root_cause = slow_performance | risk_score = 5

RULE 22 — CUSTOMER SUPPORT FAILURE
  If customer says support ignored them, gave no reply, was unhelpful after multiple contacts:
    intent = support_failure | root_cause = unresponsive_support | risk_score = 6

RULE 23 — LOGIN / ACCESS BLOCKED
  If customer cannot log in, password reset fails, or session expires repeatedly:
    intent = access_blocked | root_cause = authentication_failure | risk_score = 8
  If biometric (fingerprint/face ID) fails specifically:
    intent = access_blocked | root_cause = biometric_failure | risk_score = 8
  If blocked because too many devices registered:
    intent = access_blocked | root_cause = device_limit_exceeded | risk_score = 7

RULE 24 — UX FRICTION (works but hard to use)
  If app works but flows are confusing, UI is hard to navigate, or layout is poor:
    intent = ux_friction | root_cause = slow_performance OR offer_discoverability | risk_score = 3-4

RULE 25 — POSITIVE WITH IMPROVEMENT REQUEST
  If review is mainly positive but requests an improvement or notes a minor issue:
    sentiment = mixed | intent = feature_request
    root_cause: slow_performance | app_crash_or_bug | offer_discoverability
    risk_score = 3 or 4

Allowed labels — USE ONLY THESE exact strings:

  sentiment : {_json.dumps(VALID_SENTIMENTS)}
  intent    : {_json.dumps(VALID_INTENTS)}
  root_cause: {_json.dumps(VALID_ROOT_CAUSES)}
  risk_score: integer 0–10
  confidence: float 0.0–1.0

REQUIRED OUTPUT — valid JSON only, no markdown, no extra text:
{{
  "results": [
    {{
      "row_id": <integer>,
      "sentiment": "<label>",
      "intent": "<label>",
      "root_cause": "<label>",
      "risk_score": <0-10>,
      "cro_reasoning": "<1-2 sentence CRO justification>",
      "confidence": <0.0-1.0>
    }}
  ]
}}
""".strip()

NameError: name '_json' is not defined

In [7]:
def detect_text_column(df: pd.DataFrame) -> str:
    cols = {c.lower(): c for c in df.columns}
    for candidate in TEXT_COL_CANDIDATES:
        if candidate in cols:
            return cols[candidate]
    raise ValueError(f"No review text column found. Tried: {TEXT_COL_CANDIDATES}. Got: {list(df.columns)}")


def load_checkpoint(path: str) -> Optional[pd.DataFrame]:
    p = Path(path)
    if not p.exists() or p.stat().st_size == 0:
        return None
    try:
        checkpoint_df = pd.read_csv(path)
    except Exception as e:
        print(f"WARNING: Bad checkpoint file ({e}). Starting fresh.")
        return None
    if len(checkpoint_df) == 0:
        print("WARNING: Checkpoint is empty. Starting fresh.")
        return None
    if "source_row_id" not in checkpoint_df.columns:
        checkpoint_df = checkpoint_df.copy()
        checkpoint_df.insert(0, "source_row_id", range(len(checkpoint_df)))
    required = ["review_text", "sentiment", "intent", "root_cause", "risk_score"]
    missing = [c for c in required if c not in checkpoint_df.columns]
    if missing:
        print(f"WARNING: Checkpoint missing columns {missing}. Starting fresh.")
        return None
    return checkpoint_df


def save_checkpoint(df: pd.DataFrame, path: str) -> None:
    if df is None or len(df) == 0:
        print("WARNING: Not saving empty checkpoint.")
        return
    # Ensure directory exists for the checkpoint
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)

    # Final output is written once by run_pipeline after the full pass completes.
    # Checkpoints stay isolated so reruns cannot overwrite the base labeled file mid-run.


def append_error_log(rows: List[Dict], path: str) -> None:
    if not rows:
        return
    new_df = pd.DataFrame(rows)
    if Path(path).exists():
        try:
            old = pd.read_csv(path)
            new_df = pd.concat([old, new_df], ignore_index=True)
        except Exception:
            pass
    new_df.to_csv(path, index=False)


def extract_json_block(text: str) -> str:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text, flags=re.IGNORECASE)
        text = re.sub(r"```$", "", text.strip())
    start, end = text.find("{"), text.rfind("}")
    if start != -1 and end > start:
        return text[start:end + 1]
    return text


def normalize_result(item: Dict[str, Any]) -> Dict[str, Any]:
    sentiment  = str(item.get("sentiment",  "neutral")).lower().strip()
    intent     = str(item.get("intent",     "noise")).lower().strip()
    root_cause = str(item.get("root_cause", "other")).lower().strip()

    if not validate_label("sentiment",  sentiment):  sentiment  = "neutral"
    if not validate_label("intent",     intent):     intent     = "noise"
    if not validate_label("root_cause", root_cause): root_cause = "other"

    risk_score = item.get("risk_score", None)
    try:    risk_score = int(risk_score)
    except: risk_score = score_risk(intent, root_cause, sentiment)
    risk_score = max(0, min(10, risk_score))

    confidence = item.get("confidence", 0.0)
    try:    confidence = float(confidence)
    except: confidence = 0.0
    confidence = max(0.0, min(1.0, confidence))

    reasoning = str(item.get("cro_reasoning", "")).strip()
    if not reasoning:
        reasoning = "No reasoning provided by model."

    band_info = get_risk_band(risk_score)

    return {
        "sentiment":          sentiment,
        "intent":             intent,
        "root_cause":         root_cause,
        "risk_score":         risk_score,
        "risk_band":          band_info["band"],
        "recommended_action": band_info["action"],
        "cro_reasoning":      reasoning,
        "confidence":         confidence,
    }


def build_batch_prompt(batch_df: pd.DataFrame, text_col: str) -> str:
    reviews = []
    for _, row in batch_df.iterrows():
        reviews.append({
            "row_id":      int(row["source_row_id"]),
            "review_text": str(row[text_col]).strip(),
        })
    payload = {
        "task": "Label each customer review using CRO taxonomy. Return one result per row_id.",
        "reviews": reviews,
        "rules": [
            "Use only allowed labels from the system prompt.",
            "Match row_id exactly — every input row_id must appear in results.",
            "noise intent for gibberish/spam reviews.",
            "OTP/SMS not arriving → notification_delivery_failure root_cause.",
            "Developer options / USB debug → security_restriction_ux.",
            "Competitor named → competitor_reference root_cause.",
        ],
    }
    return json.dumps(payload, ensure_ascii=False)


In [8]:
#  PROVIDER-SPECIFIC CALLERS

def _parse_retry_wait(err_str: str) -> Optional[float]:
    """Extract seconds from 'Please try again in Xm Y.Zs' style messages."""
    match = re.search(r'try again in\s+(?:(\d+)m)?(?:([\d.]+)s)?', err_str)
    if match:
        minutes = float(match.group(1) or 0)
        seconds = float(match.group(2) or 0)
        return minutes * 60 + seconds + 5  
    return None


def _call_groq(api_key: str, user_prompt: str) -> Dict[str, Any]:
    """Single Groq call — raises on error so the outer retry loop handles it."""
    client = Groq(api_key=api_key)
    kwargs = dict(
        model=MODEL_NAME_GROQ,
        messages=[
            {"role": "system", "content": CRO_SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=0,
        response_format={"type": "json_object"},
    )
    # openai/gpt-oss-* models support a reasoning_effort knob. "low" keeps
    # output tokens down for a straightforward classification task like
    # this one, which stretches free-tier TPM/TPD budget further.
    if "gpt-oss" in MODEL_NAME_GROQ:
        kwargs["reasoning_effort"] = "low"

    response = client.chat.completions.create(**kwargs)
    raw    = response.choices[0].message.content
    parsed = json.loads(extract_json_block(raw))
    if "results" not in parsed or not isinstance(parsed["results"], list):
        raise ValueError("LLM output missing 'results' list")
    return parsed


def _call_gemini(api_key: str, user_prompt: str) -> Dict[str, Any]:
    """Single Gemini call — raises on error so the outer retry loop handles it."""
    client = genai.Client(api_key=api_key)
    response = client.models.generate_content(
        model=MODEL_NAME_GEMINI,
        contents=user_prompt,
        config=genai_types.GenerateContentConfig(
            system_instruction=CRO_SYSTEM_PROMPT,
            temperature=0,
            response_mime_type="application/json",
        ),
    )
    raw    = response.text
    parsed = json.loads(extract_json_block(raw))
    if "results" not in parsed or not isinstance(parsed["results"], list):
        raise ValueError("LLM output missing 'results' list")
    return parsed



In [9]:
#  UNIFIED CALL WITH PROVIDER ROTATION + RETRY

def call_with_retry(user_prompt: str) -> Dict[str, Any]:
    """
    Tries providers in round-robin order (Groq → Gemini-1 → Gemini-2 → ...).
    On rate-limit (429 / ResourceExhausted): sets a cooldown on that provider
    and immediately tries the next one — no wasted waiting time.
    On daily token limit: sets a long cooldown (parsed from error or 10 min).
    Falls back to exponential backoff for non-rate-limit errors.
    Raises RuntimeError only after MAX_RETRIES total attempts across all providers.
    """
    last_error = None

    for attempt in range(1, MAX_RETRIES * len(_providers) + 1):
        provider = get_next_provider()

        if provider is None:
            # All providers cooling down — find the shortest remaining cooldown
            soonest = min(_providers, key=lambda p: p["cooldown_until"])
            wait    = max(1.0, soonest["cooldown_until"] - time.time())
            print(f"  All providers in cooldown. Waiting {wait:.0f}s for [{soonest['name']}]...")
            time.sleep(wait)
            soonest["cooldown_until"] = 0.0   # reset so it's picked next
            continue

        name = provider["name"]
        key  = provider["key"]

        try:
            if name.startswith("groq"):
                result = _call_groq(key, user_prompt)
            else:
                result = _call_gemini(key, user_prompt)

            print(f"  ✓ [{name}] responded successfully")
            return result

        except Exception as e:
            last_error = e
            err_str    = str(e)

            is_429     = ("429" in err_str or "rate_limit_exceeded" in err_str
                          or "ResourceExhausted" in err_str or "RESOURCE_EXHAUSTED" in err_str)
            is_tpd     = "tokens per day" in err_str

            if is_tpd:
                wait = _parse_retry_wait(err_str) or 600
                print(f"  [{name}] Daily token limit — cooling down {wait:.0f}s")
                set_cooldown(provider, wait)
                # Don't sleep here — rotate to next provider immediately

            elif is_429:
                wait = _parse_retry_wait(err_str) or 60
                print(f"  [{name}] Rate limit (429) — cooling down {wait:.0f}s")
                set_cooldown(provider, wait)
                # Don't sleep here — rotate to next provider immediately

            else:
                # Non-rate-limit error — short backoff then retry same provider
                wait = min(MAX_SLEEP, BASE_SLEEP * (2 ** (attempt - 1))) + random.uniform(0, 1.25)
                print(f"  [{name}] Error: {e}. Retrying in {wait:.1f}s...")
                time.sleep(wait)

    raise RuntimeError(f"All providers failed after {MAX_RETRIES * len(_providers)} attempts. "
                       f"Last error: {last_error}")

In [10]:
#  LABEL BATCH 

def label_batch(batch_df: pd.DataFrame, text_col: str) -> pd.DataFrame:
    user_prompt = build_batch_prompt(batch_df, text_col)
    parsed      = call_with_retry(user_prompt)

    result_map: Dict[int, Dict] = {}
    for item in parsed["results"]:
        if not isinstance(item, dict) or "row_id" not in item:
            continue
        try:
            row_id = int(item["row_id"])
        except Exception:
            continue
        result_map[row_id] = normalize_result(item)

    labeled_rows = []
    for _, row in batch_df.iterrows():
        row_id = int(row["source_row_id"])
        base   = row.to_dict()

        if row_id in result_map:
            base.update(result_map[row_id])
        else:
            fallback = normalize_result({
                "sentiment":     "neutral",
                "intent":        "noise",
                "root_cause":    "other",
                "risk_score":    DEFAULT_RISK_SCORE,
                "cro_reasoning": "FALLBACK: model did not return this row — manual review needed.",
                "confidence":    0.0,
            })
            base.update(fallback)

        labeled_rows.append(base)

    return pd.DataFrame(labeled_rows)


In [62]:
def run_pipeline(
    df_input: pd.DataFrame,
    text_col: str,
    output_path: str = OUTPUT_CSV,
    checkpoint_path: str = CHECKPOINT_CSV,
    error_log_path: str = ERROR_LOG_CSV,
) -> pd.DataFrame:
    checkpoint_df = load_checkpoint(checkpoint_path)
    if checkpoint_df is not None and not checkpoint_df.empty:
        processed_df = checkpoint_df.copy()
        processed_ids = set(processed_df["source_row_id"].astype(int))
        remaining_df = df_input[~df_input["source_row_id"].isin(processed_ids)].copy()
        print(f"Checkpoint found — already labeled: {len(processed_df):,}")
        print(f"Remaining reviews: {len(remaining_df):,}")
    else:
        processed_df = pd.DataFrame()
        remaining_df = df_input.copy()
        print(f"No checkpoint — labeling all {len(remaining_df):,} reviews")

    labeled_batches = []
    for start in tqdm(
        range(0, len(remaining_df), BATCH_SIZE),
        desc="Labeling reviews",
        unit="batch",
    ):
        batch_df = remaining_df.iloc[start:start + BATCH_SIZE].copy()
        try:
            labeled_batch = label_batch(batch_df, text_col)
            labeled_batches.append(labeled_batch)
            processed_df = pd.concat(
                [processed_df, labeled_batch],
                ignore_index=True,
            ).drop_duplicates(
                subset=["source_row_id"],
                keep="last",
            ).sort_values("source_row_id").reset_index(drop=True)
            save_checkpoint(processed_df, checkpoint_path)
        except Exception as exc:
            error_rows = [
                {
                    "source_row_id": int(row_id),
                    "error": str(exc),
                }
                for row_id in batch_df["source_row_id"]
            ]
            append_error_log(error_rows, error_log_path)
            print(
                f"Batch starting at row {start} failed; "
                f"{len(error_rows)} rows written to {error_log_path}"
            )

    if processed_df.empty:
        raise RuntimeError("No labeled rows were produced.")

    processed_df = processed_df.sort_values("source_row_id").reset_index(drop=True)
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    processed_df.to_csv(output_path, index=False)
    print(f"Saved labeled reviews: {output_path} ({len(processed_df):,} rows)")
    return processed_df

In [63]:
#  LOAD DATA + RUN
if __name__ == "__main__":
    df_raw = pd.read_csv(INPUT_CSV)

    rename_map = {
        "content": "review", "text": "review", "score": "rating",
        "thumbsupscount": "thumbs_up_count", "at": "date",
        "appversion": "app_version", "appid": "app_name", "app": "app_name",
    }
    df_raw.rename(columns={k: v for k, v in rename_map.items() if k in df_raw.columns}, inplace=True)

    text_col = detect_text_column(df_raw)
    if text_col != "review_text":
        df_raw.rename(columns={text_col: "review_text"}, inplace=True)
        text_col = "review_text"

    df_raw[text_col] = df_raw[text_col].astype(str)
    df_raw = df_raw[df_raw[text_col].str.strip().ne("")].reset_index(drop=True)
    # Preserve the scraper's identifier when present; create it only for older inputs.
    if "source_row_id" not in df_raw.columns:
        df_raw.insert(0, "source_row_id", range(1, len(df_raw) + 1))
    else:
        df_raw["source_row_id"] = pd.to_numeric(df_raw["source_row_id"], errors="raise").astype(int)
    if "source_app" not in df_raw.columns:
        df_raw["source_app"] = ""

    print(f" Loaded {len(df_raw)} reviews | Text column: '{text_col}'")

    labeled_df = run_pipeline(df_raw, text_col)
    print(f"\nPreview ({min(5, len(labeled_df))} rows):")
    print(labeled_df[["source_app", "review_text", "sentiment", "intent",
                       "root_cause", "risk_score", "risk_band"]].head())

 Loaded 12523 reviews | Text column: 'review_text'
Checkpoint found — already labeled: 12,524
Remaining reviews: 0


Labeling reviews: 0batch [00:00, ?batch/s]

Saved labeled reviews: data/labeled/labeled_reviews.csv (12,524 rows)

Preview (5 rows):
      source_app                                        review_text sentiment  \
0  FriMi_FinTech  Still locked out of the app after 3 days! 😩 I'...  negative   
1  FriMi_FinTech  Can't create new account. Showing "run time er...  negative   
2  FriMi_FinTech            First impression "A big Run time ERROR"  negative   
3  FriMi_FinTech  poor customer hotline service in an emergency ...  negative   
4  FriMi_FinTech  I cant create an account. I tried different de...  negative   

      intent                     root_cause  risk_score risk_band  
0  complaint  notification_delivery_failure           8      HIGH  
1  complaint               app_crash_or_bug           7      HIGH  
2  complaint               app_crash_or_bug           7      HIGH  
3  complaint           unresponsive_support           8      HIGH  
4  complaint               app_crash_or_bug           7      HIGH  
